# TiniMind v2 — Training Notebook
## Indo BPE 32k + Flash Attention
**Cell 1-8:** Setup → Tokenizer → Pretrain | **Cell 9-15:** SFT → Test

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Cell 2 — Import & Config
import os, sys, math, json, random, glob, time
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from datasets import load_dataset

BASE     = '/content/drive/MyDrive/TiniMind_Prototype'
CKPT_DIR = f'{BASE}/output/checkpoints'
MC4_DIR  = f'{BASE}/output/pretrain'
SFT_DIR  = f'{BASE}/output/sft'
DATA_DIR = f'{BASE}/data/mc4_indo'
TOK_PATH = f'{BASE}/tokenizer/indo_bpe_32k.model'
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'

for d in [CKPT_DIR, MC4_DIR, SFT_DIR, DATA_DIR, f'{BASE}/tokenizer']:
    os.makedirs(d, exist_ok=True)

sys.path.insert(0, BASE)
print(f'Device: {DEVICE}')

Device: cuda


In [3]:
# Cell 3 — Pretrain Config & Data Loader
import subprocess
from google.colab import drive
from model_v2 import TiniMind
from config import ModelConfig

SEQ_LEN  = 1024
BATCH_PT = 4
GRAD_ACC = 8
LR_MAX   = 3e-4
LR_MIN   = 1e-5
WARMUP   = 200
LOG_STEP = 100
SAVE_STEP= 1000
MAX_STEP = 20000

def get_lr(step):
    if step < WARMUP: return LR_MAX * (step+1) / WARMUP
    p = (step-WARMUP) / max(1, MAX_STEP-WARMUP)
    return LR_MIN + (LR_MAX-LR_MIN) * 0.5 * (1 + math.cos(math.pi*p))

def remount_drive():
    subprocess.run(['fusermount', '-uz', '/content/drive'], capture_output=True)
    drive.mount('/content/drive', force_remount=True)
    print("Drive remounted!")

def get_batch(files):
    for attempt in range(3):
        try:
            data = np.fromfile(random.choice(files), dtype=np.uint16).astype(np.int64)
            if len(data) <= SEQ_LEN+1: return None, None
            ix = np.random.randint(0, len(data)-SEQ_LEN-1, size=BATCH_PT)
            x  = torch.stack([torch.from_numpy(data[i:i+SEQ_LEN])     for i in ix]).to(DEVICE)
            y  = torch.stack([torch.from_numpy(data[i+1:i+SEQ_LEN+1]) for i in ix]).to(DEVICE)
            return x, y
        except OSError as e:
            if e.errno == 107:
                print(f"Drive disconnect! Remounting... (attempt {attempt+1}/3)")
                remount_drive()
            else:
                raise e
    return None, None

bin_files = sorted(glob.glob(f'{DATA_DIR}/chunk_*.bin'))
total_tok = sum(os.path.getsize(f)//2 for f in bin_files)
print(f'Chunks: {len(bin_files)} | Total: {total_tok/1e9:.2f}B token')
if not bin_files: print('Belum ada data! Jalankan Cell 5 dulu.')

Chunks: 301 | Total: 3.00B token


In [4]:
cfg = ModelConfig(
    num_layers   = 24,
    hidden_size  = 1024,
    num_heads    = 16,
    num_kv_heads = 4,
    vocab_size   = 32000,
    max_seq_len  = SEQ_LEN,
    dropout      = 0.0,
)

ckpts      = sorted(glob.glob(f'{MC4_DIR}/step_*.pt'))
start_step = 0
model      = TiniMind(cfg).to(DEVICE)

if ckpts:
    ck = torch.load(ckpts[-1], map_location=DEVICE, weights_only=False)
    model.load_state_dict(ck['model_state'])
    start_step = ck['step']
    print(f'Resume dari step {start_step}')
else:
    print('Init model baru')

print(f'Params : {model.num_params()/1e6:.1f}M')
print(f'Config : {cfg.num_layers}L x {cfg.hidden_size}H | {cfg.num_heads}Q/{cfg.num_kv_heads}KV')

Flash Attention aktif (F.scaled_dot_product_attention) | vocab=32000
Resume dari step 3000
Params : 270.6M
Config : 24L x 1024H | 16Q/4KV


In [ ]:
# Cell 5 — Pretrain Training Loop (+ val loss + grafik)
import matplotlib.pyplot as plt

assert bin_files, 'Tidak ada data! Jalankan Cell 4 dulu.'

# Pisah 2 chunk terakhir untuk validasi
val_files   = bin_files[-2:]
train_files = bin_files[:-2]
print(f'Train: {len(train_files)} chunks | Val: {len(val_files)} chunks')

opt    = torch.optim.AdamW(model.parameters(), lr=LR_MAX, weight_decay=0.1)
scaler = torch.amp.GradScaler('cuda')
if ckpts:
    if 'optimizer_state' in ck: opt.load_state_dict(ck['optimizer_state'])
    if 'scaler_state'    in ck: scaler.load_state_dict(ck['scaler_state'])

@torch.no_grad()
def val_loss(n_batches=20):
    model.eval()
    total = 0
    for _ in range(n_batches):
        x, y = get_batch(val_files)
        if x is None: continue
        with torch.amp.autocast('cuda'):
            _, loss, _ = model(x, y)
        total += loss.item()
    model.train()
    return total / n_batches

step = start_step; accum = 0
train_log = []   # (step, train_loss)
val_log   = []   # (step, val_loss)
model.train(); opt.zero_grad()

print(f'Mulai step {start_step} -> {start_step+MAX_STEP}\n')

while step < start_step + MAX_STEP:
    x, y = get_batch(train_files)
    if x is None: continue
    with torch.amp.autocast('cuda'):
        _, loss, _ = model(x, y)
    scaler.scale(loss / GRAD_ACC).backward()
    accum += 1
    if accum % GRAD_ACC == 0:
        step += 1
        lr = get_lr(step - start_step)
        for pg in opt.param_groups: pg['lr'] = lr
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad()

        if step % LOG_STEP == 0:
            vl = val_loss()
            print(f'step {step:6d} | train: {loss.item():.4f} | val: {vl:.4f} | lr: {lr:.2e}')
            train_log.append((step, loss.item()))
            val_log.append((step, vl))

        if step % SAVE_STEP == 0:
            p = f'{MC4_DIR}/step_{step:07d}_loss_{loss.item():.4f}.pt'
            torch.save({'step': step, 'config': cfg,
                        'model_state': model.state_dict(),
                        'optimizer_state': opt.state_dict(),
                        'scaler_state': scaler.state_dict()}, p)
            print(f'  >> Checkpoint saved: {p}')

# ── Plot loss setelah selesai ─────────────────────────────────────
if train_log:
    steps_t, losses_t = zip(*train_log)
    steps_v, losses_v = zip(*val_log)

    plt.figure(figsize=(10, 4))
    plt.plot(steps_t, losses_t, label='Train Loss', alpha=0.8)
    plt.plot(steps_v, losses_v, label='Val Loss',   alpha=0.8)
    plt.xlabel('Step'); plt.ylabel('Loss')
    plt.title('TiniMind Pretrain Loss')
    plt.legend(); plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{MC4_DIR}/loss_curve.png', dpi=120)
    plt.show()
    print(f'Grafik disimpan: {MC4_DIR}/loss_curve.png')

with open(f'{MC4_DIR}/log.jsonl', 'a') as f:
    for s, l in train_log: f.write(json.dumps({'step': s, 'loss': round(l,4)}) + '\n')
print(f'Selesai di step {step}.')

Train: 299 chunks | Val: 2 chunks
Mulai step 3000 -> 23000

Drive disconnect! Remounting... (attempt 1/3)
Mounted at /content/drive
Drive remounted!
step   3100 | train: 3.9109 | val: 3.8084 | lr: 1.51e-04
step   3200 | train: 3.7588 | val: 3.8238 | lr: 3.00e-04
step   3300 | train: 3.9794 | val: 3.8502 | lr: 3.00e-04
step   3400 | train: 3.7162 | val: 3.8941 | lr: 3.00e-04


---
## Bagian 2 — SFT
Jalankan setelah pretrain loss < 3.5

In [ ]:
# Cell 9 — SFT Config
MAX_SEQ  = 512
BATCH_SF = 1
GRAD_SF  = 4
LR_SF    = 2e-6
WARMUP_SF= 100
KL_BETA  = 0.1    # KL penalty, 0 = off
MIX_PT   = 0.15   # 15% batch dari mC4 (data mixing)
EVAL_INT = 200
PATIENCE = 5

BAD_STARTS = (':', '1.', '2.', '- ', '* ', '#', '/')

def get_lr_sf(step, total):
    min_lr = LR_SF * 0.1
    if step < WARMUP_SF: return LR_SF * (step+1) / WARMUP_SF
    p = (step-WARMUP_SF) / max(1, total-WARMUP_SF)
    return min_lr + (LR_SF-min_lr) * 0.5 * (1 + math.cos(math.pi*p))

print(f'KL beta: {KL_BETA} | Mix pretrain: {MIX_PT*100:.0f}%')

In [ ]:
# Cell 10 — Load SFT Dataset
all_examples = []

def parse_sharegpt(ds):
    out = []
    for ex in ds:
        turns = []
        for msg in ex.get('conversations',[]):
            role = 'user' if msg.get('from')=='human' else 'assistant'
            c    = msg.get('value','').strip()
            if c: turns.append((role,c))
        if turns and turns[-1][0]=='assistant': out.append({'turns':turns})
    return out

for name, loader in [
    ('sharegpt-indonesian',      lambda: parse_sharegpt(load_dataset('FreedomIntelligence/sharegpt-indonesian',split='train'))),
    ('evol-instruct-indonesian', lambda: parse_sharegpt(load_dataset('FreedomIntelligence/evol-instruct-indonesian',split='train'))),
]:
    try:
        p = loader(); all_examples.extend(p); print(f'OK {name}: {len(p)}')
    except Exception as e: print(f'FAIL {name}: {e}')

try:
    ds = load_dataset('CohereForAI/aya_dataset',split='train').filter(lambda x: x['language']=='Indonesian')
    for ex in ds:
        inp = ex.get('inputs','').strip(); out = ex.get('targets','').strip()
        if inp and out and not out.startswith(BAD_STARTS):
            all_examples.append({'turns':[('user',inp),('assistant',out)]})
    print(f'OK aya_dataset: {len(ds)}')
except Exception as e: print(f'FAIL aya: {e}')

# sft_generated.jsonl — buatan Claude (upload ke Drive dulu)
gen = f'{BASE}/dataset/sft_generated.jsonl'
if os.path.exists(gen):
    with open(gen) as f:
        added = sum(1 for l in f if l.strip() and all_examples.append(json.loads(l)) is None)
    print(f'OK sft_generated: {added}')
else:
    print(f'sft_generated.jsonl tidak ada di {gen} — upload ke Drive dulu')

random.shuffle(all_examples)
print(f'\nTotal: {len(all_examples)} examples')

In [ ]:
# Cell 11 — SFT Dataset Class
def build_input_labels(ex):
    ids, lbls = [], []
    for role, content in ex['turns']:
        if role == 'user':
            t = tok.encode(f'<penggunna>\n{content.strip()}\n</penggunna>\n<asisten>\n')
            ids.extend(t); lbls.extend([-100]*len(t))
        else:
            t = tok.encode(content.strip()+'\n') + [EOT_ID]
            ids.extend(t); lbls.extend(t)
    return ids, lbls

class SFTDataset(Dataset):
    def __init__(self, exs):
        self.data=[]; skip=0
        for ex in exs:
            ids, lbls = build_input_labels(ex)
            if len(ids) > MAX_SEQ or not any(l!=-100 for l in lbls): skip+=1; continue
            self.data.append((ids,lbls))
        print(f'  {len(self.data)} ok | {skip} skip')
    def __len__(self): return len(self.data)
    def __getitem__(self,i): return self.data[i]

def collate(batch):
    X = torch.zeros(len(batch), MAX_SEQ, dtype=torch.long)
    Y = torch.full((len(batch), MAX_SEQ), -100, dtype=torch.long)
    for i,(ids,lbls) in enumerate(batch):
        L = min(len(ids), MAX_SEQ)
        X[i,:L] = torch.tensor(ids[:L])
        Y[i,:L] = torch.tensor(lbls[:L])
    return X, Y

sp   = int(len(all_examples)*0.95)
tds  = SFTDataset(all_examples[:sp])
vds  = SFTDataset(all_examples[sp:])
tdl  = DataLoader(tds, batch_size=BATCH_SF, shuffle=True,  collate_fn=collate)
vdl  = DataLoader(vds, batch_size=BATCH_SF, shuffle=False, collate_fn=collate)
print(f'Train: {len(tds)} | Val: {len(vds)}')

In [ ]:
# Cell 12 — Load Model SFT + Ref Model
ckpts = sorted(glob.glob(f'{MC4_DIR}/step_*.pt'))
assert ckpts, 'Belum ada checkpoint pretrain! Jalankan Cell 7-8 dulu.'

raw = torch.load(ckpts[-1], map_location=DEVICE, weights_only=False)
cfg = raw['config']
print(f'Load checkpoint: {ckpts[-1]}')

sft_model = TiniMind(cfg).to(DEVICE)
sft_model.load_state_dict(raw['model_state'])

ref_model = TiniMind(cfg).to(DEVICE)
ref_model.load_state_dict(raw['model_state'])
for p in ref_model.parameters(): p.requires_grad = False
ref_model.eval()

sft_model.train()
print(f'SFT model : {sft_model.num_params()/1e6:.1f}M params')
print(f'Ref model : frozen (untuk KL penalty)')

In [ ]:
# Cell 13 — Helper: evaluate + generate
@torch.no_grad()
def evaluate(model, loader):
    model.eval(); tot_loss=tot_tok=0
    for X,Y in loader:
        X,Y = X.to(DEVICE), Y.to(DEVICE)
        if not (Y!=-100).any(): continue
        logits,_,_ = model(X)
        loss = F.cross_entropy(logits.view(-1,logits.size(-1)), Y.view(-1), ignore_index=-100, reduction='sum')
        tot_loss += loss.item(); tot_tok += (Y!=-100).sum().item()
    model.train()
    return tot_loss / max(1, tot_tok)

@torch.no_grad()
def generate(model, prompt, temp=0.7, top_k=40, max_new=150):
    model.eval()
    pfx = tok.encode(f'<penggunna>\n{prompt}\n</penggunna>\n<asisten>\n')
    ids = torch.tensor([pfx], dtype=torch.long, device=DEVICE)
    seen = {}
    for _ in range(max_new):
        logits = model(ids)[0][0,-1] / temp
        for t,c in seen.items(): logits[t] -= 0.3*c
        top = torch.topk(logits, top_k)
        logits[logits < top.values[-1]] = float('-inf')
        nxt = torch.multinomial(torch.softmax(logits,-1),1).item()
        if nxt == EOT_ID: break
        seen[nxt] = seen.get(nxt,0)+1
        ids = torch.cat([ids, torch.tensor([[nxt]],device=DEVICE)], dim=1)
    model.train()
    return tok.decode(ids[0,len(pfx):].tolist())

print('Helper siap.')

In [ ]:
# Cell 14 — SFT Training Loop
opt_sf   = torch.optim.AdamW(sft_model.parameters(), lr=LR_SF, weight_decay=0.1)
total_sf = len(tdl) // GRAD_SF
best_val = float('inf'); patience = 0; step = 0; logs = []
bin_files= sorted(glob.glob(f'{DATA_DIR}/chunk_*.bin'))

print(f'SFT | {total_sf} steps | KL={KL_BETA} | Mix={MIX_PT*100:.0f}%\n')
sft_model.train(); opt_sf.zero_grad()

for bidx,(sft_x, sft_y) in enumerate(tdl):
    sft_x = sft_x.to(DEVICE); sft_y = sft_y.to(DEVICE)

    # Data mixing: sesekali pakai batch dari pretrain mC4
    if bin_files and random.random() < MIX_PT:
        data = np.fromfile(random.choice(bin_files), dtype=np.uint16).astype(np.int64)
        if len(data) > MAX_SEQ+1:
            i  = np.random.randint(0, len(data)-MAX_SEQ-1)
            px = torch.from_numpy(data[i:i+MAX_SEQ]).unsqueeze(0).to(DEVICE)
            py = torch.from_numpy(data[i+1:i+MAX_SEQ+1]).unsqueeze(0).to(DEVICE)
            logits,_,_ = sft_model(px)
            sft_loss = F.cross_entropy(logits.view(-1,logits.size(-1)), py.view(-1))
        else:
            logits,sft_loss,_ = sft_model(sft_x, sft_y) ; logits=logits
    else:
        logits, sft_loss, _ = sft_model(sft_x, sft_y)

    # KL penalty vs ref model
    with torch.no_grad():
        ref_logits,_,_ = ref_model(sft_x)
    kl_loss = F.kl_div(F.log_softmax(sft_model(sft_x)[0],-1),
                        F.softmax(ref_logits,-1), reduction='batchmean')
    loss = sft_loss + KL_BETA * kl_loss
    (loss / GRAD_SF).backward()

    if (bidx+1) % GRAD_SF == 0:
        step += 1
        for pg in opt_sf.param_groups: pg['lr'] = get_lr_sf(step, total_sf)
        torch.nn.utils.clip_grad_norm_(sft_model.parameters(), 1.0)
        opt_sf.step(); opt_sf.zero_grad()

        if step % 50 == 0:
            print(f'step {step:4d}/{total_sf} | sft={sft_loss.item():.4f} kl={kl_loss.item():.4f}')

        if step % EVAL_INT == 0 or step == total_sf:
            vl     = evaluate(sft_model, vdl)
            sample = generate(sft_model, 'Apa itu kecerdasan buatan?')
            print(f'\n{"─"*55}')
            print(f'Step {step} | val_loss: {vl:.4f}')
            print(f'Sample: {sample[:200]}')
            logs.append({'step':step,'val_loss':round(vl,4),'sample':sample[:200]})
            torch.save({'step':step,'model_state_dict':sft_model.state_dict(),'val_loss':vl},
                       f'{SFT_DIR}/ckpt_step{step:04d}_vl{vl:.4f}.pt')
            if vl < best_val:
                best_val=vl; patience=0
                torch.save(sft_model.state_dict(), f'{SFT_DIR}/sft_best.pt')
                print(f'New best: {vl:.4f}')
            else:
                patience += 1
                if patience >= PATIENCE: print(f'Early stop. Best: {best_val:.4f}'); break
            print(f'{"─"*55}\n')

with open(f'{SFT_DIR}/log.json','w') as f: json.dump(logs,f,indent=2,ensure_ascii=False)
print(f'\nSFT selesai! Best val_loss: {best_val:.4f}')

In [ ]:
# Cell 15 — Test Model
def chat(prompt, temp=0.7, max_new=200):
    print(f'[User] {prompt}')
    print(f'[AI]   {generate(sft_model, prompt, temp=temp, max_new=max_new)}')
    print()

chat('Halo, siapa kamu?')
chat('Apa itu kecerdasan buatan?')
chat('Berapa hasil dari 12 dikali 8?')
chat('Kenapa langit berwarna biru?')
chat('Ceritakan tentang budaya Indonesia.')